# 04 — Statistical Analysis

With clean data in hand, we run the core statistical test.

This notebook covers:

1. **Computing conversion rates** per group
2. **The two-proportion z-test** — why it, how it works
3. **Interpreting p-values** — what they mean and what they do NOT mean
4. **Confidence intervals** — a richer picture than a single p-value
5. **Visualizing the result** — sampling distribution and CI plots

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

sys.path.append("../src")
from ab_stats import two_proportion_ztest, confidence_interval

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

ALPHA = 0.05

df = pd.read_csv("../data/ab_data_clean.csv")
print(f"Loaded: {len(df):,} rows")
df.head(3)

## 1. Conversion Rates

In [ ]:
grp = df.groupby("group")["converted"].agg(conversions="sum", users="count")
grp["rate"] = grp["conversions"] / grp["users"]

ctrl = grp.loc["control"]
trt  = grp.loc["treatment"]

print(f"Control   — users: {ctrl.users:>8,}  conversions: {ctrl.conversions:>7,}  rate: {ctrl.rate:.4%}")
print(f"Treatment — users: {trt.users:>8,}  conversions: {trt.conversions:>7,}  rate: {trt.rate:.4%}")
print(f"Absolute difference: {trt.rate - ctrl.rate:+.4%}")
print(f"Relative difference: {(trt.rate - ctrl.rate) / ctrl.rate:+.2%}")

## 2. The Two-Proportion Z-Test

We use a **two-proportion z-test** because:
- The outcome is binary (converted: 0 or 1)
- Sample sizes are large (N >> 30), so the Central Limit Theorem applies
- We have two independent groups

### How it works

Under H₀ (no difference), the test statistic is:

$$z = \frac{\hat{p}_T - \hat{p}_C}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_C} + \frac{1}{n_T}\right)}}$$

where $\hat{p}$ is the **pooled proportion** — the overall conversion rate across both groups.

If H₀ were true, this z-statistic would follow a standard normal distribution N(0,1).
We check how extreme our observed z is under that distribution.

In [ ]:
z, p_val, p_c, p_t, lift_abs, lift_rel = two_proportion_ztest(
    ctrl.conversions, ctrl.users,
    trt.conversions,  trt.users,
)

print("=" * 45)
print("  Two-Proportion Z-Test Results")
print("=" * 45)
print(f"  Control rate   : {p_c:.4%}")
print(f"  Treatment rate : {p_t:.4%}")
print(f"  Absolute lift  : {lift_abs:+.4%}")
print(f"  Relative lift  : {lift_rel:+.2%}")
print(f"  z-statistic    : {z:.4f}")
print(f"  p-value        : {p_val:.4f}")
print(f"  alpha          : {ALPHA}")
print()
if p_val < ALPHA:
    print("  RESULT: Reject H0 — statistically significant")
else:
    print("  RESULT: Fail to reject H0 — not statistically significant")
print("=" * 45)

## 3. Interpreting the p-value

The p-value is one of the most misunderstood numbers in science.

### What p-value IS

> The probability of observing a result **at least as extreme as ours**, *assuming H₀ is true*.

In plain English: if the new page truly had zero effect, how often would random chance
produce a difference this large or larger? A large p-value means "often" — the data is
consistent with there being no real effect.

### What p-value is NOT

| Misconception | Reality |
|---|---|
| p = 0.19 means 19% chance the null is true | p-value says nothing about P(H₀ is true) |
| p < 0.05 means the effect is real | It means the data is unlikely under H₀; the effect could still be noise |
| p > 0.05 means no effect exists | "Absence of evidence is not evidence of absence" |
| A smaller p-value means a bigger effect | p-value conflates effect size and sample size |
| p-value tells us the effect is practically meaningful | That requires effect size + business context |

**The p-value is a measure of surprise, not a measure of importance.**

In [ ]:
# Visualize the null distribution and where our test statistic falls
x = np.linspace(-4, 4, 400)
y = stats.norm.pdf(x)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x, y, color="steelblue", linewidth=2, label="N(0,1) under H₀")

# Shade rejection regions (two-tailed)
crit = stats.norm.ppf(1 - ALPHA / 2)
ax.fill_between(x, y, where=(x <= -crit), color="tomato", alpha=0.3, label=f"Rejection region (α={ALPHA})")
ax.fill_between(x, y, where=(x >=  crit), color="tomato", alpha=0.3)

# Mark observed z
ax.axvline(z, color="darkgreen", linewidth=2, linestyle="--", label=f"Observed z = {z:.2f}")
ax.axvline(-crit, color="tomato", linewidth=1, linestyle=":")
ax.axvline( crit, color="tomato", linewidth=1, linestyle=":")

ax.set_xlabel("z-statistic", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Sampling distribution under H₀ — where does our z land?", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Critical value: ±{crit:.2f}")
inside_outside = "INSIDE" if abs(z) < crit else "OUTSIDE"
print(f"Our z = {z:.2f} is {inside_outside} the non-rejection region")

## 4. Confidence Intervals

A confidence interval gives a **range of plausible values** for the true conversion rate.
It is more informative than a single point estimate.

We use the **Wilson score interval** — it works well even near 0 or 1 and for
moderate sample sizes, unlike the simpler normal approximation interval.

A 95% CI means: if we ran this experiment 100 times and computed a CI each time,
95 of those intervals would contain the true population proportion.
*(It does NOT mean "95% chance the true value is in this specific interval.")*

In [ ]:
ci_c = confidence_interval(ctrl.conversions, ctrl.users, ALPHA)
ci_t = confidence_interval(trt.conversions,  trt.users,  ALPHA)

print(f"Control   95% CI: [{ci_c[0]:.4%}, {ci_c[1]:.4%}]  (point: {p_c:.4%})")
print(f"Treatment 95% CI: [{ci_t[0]:.4%}, {ci_t[1]:.4%}]  (point: {p_t:.4%})")

# Plot
fig, ax = plt.subplots(figsize=(7, 3.5))
groups = ["Control", "Treatment"]
means  = [p_c, p_t]
errs   = [[m - lo for m, (lo, _) in zip(means, [ci_c, ci_t])],
          [hi - m for m, (_, hi) in zip(means, [ci_c, ci_t])]]

ax.barh(groups, [m * 100 for m in means], xerr=[[e*100 for e in errs[0]], [e*100 for e in errs[1]]],
        color=["steelblue", "coral"], capsize=6, height=0.4)
ax.set_xlabel("Conversion rate (%)")
ax.set_title("Conversion rates with 95% confidence intervals")
ax.set_xlim(10, 14)
plt.tight_layout()
plt.show()

## 5. Visualizing the Difference

Instead of just looking at each group separately, we can compute a CI directly on
the **difference** between treatment and control. This is what we actually care about.

In [ ]:
# CI on the difference using normal approximation
se_diff = np.sqrt(p_c*(1-p_c)/ctrl.users + p_t*(1-p_t)/trt.users)
z_crit  = stats.norm.ppf(1 - ALPHA/2)
diff_lo = lift_abs - z_crit * se_diff
diff_hi = lift_abs + z_crit * se_diff

print(f"Observed difference: {lift_abs:+.4%}")
print(f"95% CI on diff:      [{diff_lo:+.4%}, {diff_hi:+.4%}]")
print()
contains_zero = diff_lo < 0 < diff_hi
h0_label = "consistent with H0" if contains_zero else "excludes H0"
print(f"CI contains zero: {contains_zero} — {h0_label}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.axvline(0, color="gray", linewidth=1.2, linestyle="--", label="No effect (H₀)")
ax.errorbar(lift_abs * 100, 0.5, xerr=[[abs(diff_lo-lift_abs)*100], [abs(diff_hi-lift_abs)*100]],
            fmt="o", color="steelblue", capsize=8, markersize=8, linewidth=2,
            label=f"Observed lift + 95% CI")
ax.set_xlabel("Difference in conversion rate (percentage points)")
ax.set_yticks([])
ax.set_title("95% CI on treatment vs. control difference")
ax.legend()
ax.set_xlim(-0.5, 0.5)
plt.tight_layout()
plt.show()

## Summary

| Metric | Value |
|---|---|
| Control conversion rate | ~11.9% |
| Treatment conversion rate | ~11.7% |
| Absolute difference | ~−0.2pp |
| p-value | ~0.19 |
| 95% CI on difference | contains zero |
| Decision | Fail to reject H₀ |

**We cannot conclude the new landing page improves conversion.** The observed difference
is small and consistent with random variation.

But statistical significance alone does not tell the whole story.
Proceed to **[05 — Practical Significance](05_practical_significance.ipynb)**
to understand *how big* an effect we could have detected.